# Higra Max-Tree with `mmcfilters.Attribute.MAX_DIST`

This notebook builds a max-tree with Higra from `../dat/imgObjetos.png`, computes the `MAX_DIST` attribute with `mmcfilters`, and then applies two filters with Higra:

- a direct attribute-threshold filter;
- an extinction-value filter driven by the same `MAX_DIST` values.


## Imports and plotting helpers

The notebook keeps Higra responsible for hierarchy construction and image reconstruction. `mmcfilters` is used only for the attribute that is not available in Higra.


In [ ]:
import mmcfilters
from pathlib import Path

import cv2 as cv
import higra as hg
import matplotlib.pyplot as plt
import numpy as np


def load_grayscale(path):
    image = cv.imread(str(path), cv.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(path)
    return np.ascontiguousarray(image, dtype=np.uint8)


def show_images(images, *, figsize=(16, 5), vmin=0, vmax=255):
    fig, axes = plt.subplots(1, len(images), figsize=figsize, constrained_layout=True)
    if len(images) == 1:
        axes = [axes]
    for ax, (title, image) in zip(axes, images):
        ax.imshow(image, cmap="gray", vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.axis("off")
    plt.show()


## Load the input image

The image is loaded as `uint8` grayscale data. Its flattened pixel order is the leaf order used by Higra grid component trees.


In [ ]:
image_path = Path("../dat/imgObjetos.png")
input_image = load_grayscale(str(image_path))
if input_image is None:
    raise FileNotFoundError(image_path)

rows, cols = input_image.shape
print(f"image shape: {input_image.shape}, dtype: {input_image.dtype}")

show_images([("Input image", input_image)])


## Build the max-tree with Higra

`hg.get_8_adjacency_graph` uses the 8-neighborhood on the image grid. This is the Higra counterpart of the `mmcfilters` adjacency radius `1.5`, which connects horizontal, vertical, and diagonal neighbors.


In [ ]:
higra_graph = hg.get_8_adjacency_graph(input_image.shape)
higra_tree, higra_altitude = hg.component_tree_max_tree(
    higra_graph,
    input_image.ravel(),
)

higra_parent = np.asarray(higra_tree.parents(), dtype=np.int32)
higra_altitude = np.asarray(higra_altitude, dtype=np.uint8)

print(f"Higra vertices: {higra_tree.num_vertices()}")
print(f"Higra leaves:   {higra_tree.num_leaves()}")
print(f"Higra root:     {higra_tree.root()}")


## Import the Higra hierarchy into `mmcfilters`

`createFromHigraParent` preserves the imported Higra node-id domain. This lets `mmcfilters` compute attributes and return them directly aligned with `higra_tree` when `outputSpace=NodeIdSpace.HIGRA` is requested.


In [ ]:
adjacency_radius = 1.5
weighted_tree = mmcfilters.MorphologicalTreeFactory.createFromHigraParent(
    higra_parent.tolist(),
    higra_altitude.tolist(),
    rows,
    cols,
    mmcfilters.MorphologicalTreeKind.MAX_TREE,
    radius=adjacency_radius,
)

roundtrip_image = weighted_tree.reconstructionImage()
assert np.array_equal(roundtrip_image, input_image)
assert weighted_tree.numHigraNodes == higra_tree.num_vertices()

print(f"mmcfilters internal node slots: {weighted_tree.numInternalNodeSlots}")
print("round-trip reconstruction matches the input image")


## Compute `MAX_DIST` with `mmcfilters`

The result is indexed by Higra node id, not by the internal `mmcfilters` node id. Leaves and internal nodes therefore share the same layout as `higra_parent` and `higra_altitude`.

For Higra component trees, the leaves are the original pixels. A one-pixel component has `MAX_DIST = 0.0`, so the leaf slots are explicitly set to that unit value before the vector is reused by Higra.


In [ ]:
max_dist_by_higra = np.asarray(
    mmcfilters.Attribute.computeSingleAttribute(weighted_tree, mmcfilters.Attribute.MAX_DIST, outputSpace=mmcfilters.NodeIdSpace.HIGRA)
)

assert max_dist_by_higra.shape[0] == higra_tree.num_vertices()

# Higra leaves are single pixels. The MAX_DIST unit value for a one-pixel
# component is 0.0, and Higra attribute/extinction routines expect finite input.
max_dist_by_higra[:higra_tree.num_leaves()] = 0.0
assert np.isfinite(max_dist_by_higra).all()

internal_max_dist = max_dist_by_higra[higra_tree.num_leaves():]
positive_internal_max_dist = internal_max_dist[internal_max_dist > 0]

print(f"MAX_DIST shape: {max_dist_by_higra.shape}")
print(f"MAX_DIST range: {max_dist_by_higra.min():.1f} to {max_dist_by_higra.max():.1f}")
print(f"positive internal nodes: {positive_internal_max_dist.size}")


## Direct attribute filtering with Higra

The direct filter deletes nodes whose `MAX_DIST` is below a chosen threshold and then asks Higra to reconstruct the leaf data. The root is always preserved so the reconstruction remains valid.


In [ ]:
attribute_threshold=10**2
attribute_filtered_image = hg.reconstruct_leaf_data(higra_tree, higra_altitude, max_dist_by_higra < attribute_threshold).reshape(input_image.shape)

show_images(
    [
        ("Input image", input_image),
        (f"MAX_DIST >= {attribute_threshold:.0f}", attribute_filtered_image)
    ],
    figsize=(18, 6),
)